# 08 · Evaluation & Ablations

**Goal:** turn the branch graph from notebook 07 into the metrics the
final-report rubric grades:

1. **Primary branch count recall** — fraction of tape-measured ground-truth
   branches that the pipeline detected.
2. **Attachment angle MAE** (deg) — mean absolute error between predicted and
   measured branch angle off the trunk axis.
3. **Attachment height MAE** (m) — same, for branch attachment height above the
   trunk base.

Plus the three ablations the proposal called for, computed by re-running
notebooks 04 / 06 / 07 with different upstream settings and collating CSVs:

- Classical (Canny + brightness) vs. LAB bark segmentation
- Two-view (notebook 03) vs. multi-view (notebook 04) reconstruction
- Reprojection filter on (notebook 06) vs. off (raw COLMAP cloud)

This notebook is **stateful across runs**: each scoring writes a row to
`outputs/metrics/<run_label>.csv`. The ablation cells at the bottom just
collate those CSVs into a comparison table — no recomputation.

### Important: COLMAP units are NOT metric by default

The branch graph from notebook 07 is in COLMAP's arbitrary world units, not
meters. Ground-truth measurements (tape, protractor) are in metres and degrees.
**Section 2 of this notebook handles the scale calibration** — you measure the
actual trunk height in metres once, the notebook computes the scale factor,
and all subsequent comparisons happen in metric units.


In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline


## 0. Configuration

In [ ]:
from src import evaluate, sfm, viz

TREE_ID    = "tree_5"
RUN_LABEL  = "lab_filter_on"   # short tag identifying THIS configuration; metrics row is keyed by it.

GT_DIR        = PROJECT_ROOT / "data" / "ground_truth"
RECON_DIR     = PROJECT_ROOT / "outputs" / "reconstructions"
METRICS_DIR   = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_PATH = RECON_DIR / f"{TREE_ID}_graph.npz"
GT_CSV     = GT_DIR / f"{TREE_ID}.csv"
OUT_CSV    = METRICS_DIR / f"{RUN_LABEL}.csv"

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f"Branch graph not found at {GRAPH_PATH}. Run notebook 07 first."
    )

print(f"Tree:      {TREE_ID}")
print(f"Run label: {RUN_LABEL}")
print(f"Graph:     {GRAPH_PATH}")
print(f"GT CSV:    {GT_CSV}  ({'exists' if GT_CSV.exists() else 'NOT YET — see section 3'})")


## 1. Load the branch graph + extract predicted primary branches

A "primary branch" is an edge that transitions from the trunk to off-trunk
(parent is on the trunk line, child is not). `evaluate.predicted_branches_from_graph`
walks the graph and emits one PredBranch per such edge, with the height and
angle measured against the trunk axis.

All values here are in **COLMAP units** until section 2 applies the scale.


In [ ]:
# Load the graph saved by notebook 07.
data = np.load(GRAPH_PATH, allow_pickle=True)
nodes           = data["nodes"]
edges_arr       = data["edges"]
trunk_root      = data["trunk_root"]
trunk_direction = data["trunk_direction"]
parents_arr     = data["parents"]

edges     = [(int(p), int(c)) for p, c in edges_arr]
parent_of = {int(c): int(p) for c, p in parents_arr}

print(f"Loaded graph: {len(nodes)} nodes, {len(edges)} edges")
print(f"Trunk root (COLMAP units): {trunk_root}")
print(f"Trunk direction:           {trunk_direction}")

# Extract primary branches in COLMAP units.
preds_colmap = evaluate.predicted_branches_from_graph(
    nodes=nodes,
    edges=edges,
    trunk_root=trunk_root,
    trunk_axis=trunk_direction,
    parent_of=parent_of,
)
print(f"\nDetected {len(preds_colmap)} primary branches (COLMAP units):")
print(f"{'id':>3}  {'height (unit)':>14}  {'angle (deg)':>11}")
for b in preds_colmap:
    print(f"{b.branch_id:>3}  {b.attach_height_m:>14.3f}  {b.attach_angle_deg:>11.1f}")


## 2. Scale calibration — COLMAP units → metres

Measure the actual height of the trunk from base to first major branching fork,
in metres. Use a tape measure or estimate by stadia (if you stand a known
distance away). Enter that value in `MEASURED_TRUNK_HEIGHT_M` below.

The notebook computes the **scale factor** = `measured_metres / colmap_units`,
then converts every predicted height from COLMAP units to metres. Angles are
unitless so they pass through unchanged.

If you can't measure the trunk easily (tree is too tall / on a slope), measure
*any* distance you can — between two prominent features visible in the cloud —
and substitute the appropriate values below.


In [ ]:
# CALIBRATION — edit these two values once you have a real-world measurement.
MEASURED_TRUNK_HEIGHT_M = None  # e.g. 3.5  — metres from trunk base to first fork
# Optional: if you'd rather calibrate against trunk diameter or an inter-branch
# distance, set MEASURED_TRUNK_HEIGHT_M to that value and update the COLMAP-side
# measurement below accordingly. This pair just needs to be a known scale pair.

# COLMAP-side measurement: trunk height in COLMAP units, from the cloud.
# Computed as the inlier extent along the trunk axis from notebook 07's RANSAC.
# Re-derive here so the calibration is self-contained.
projs = (nodes - trunk_root) @ trunk_direction
trunk_extent_colmap = float(projs.max() - projs.min())
print(f"Trunk extent in COLMAP units (max - min projection on axis): {trunk_extent_colmap:.3f}")

if MEASURED_TRUNK_HEIGHT_M is None:
    print()
    print("MEASURED_TRUNK_HEIGHT_M is None — scale calibration skipped.")
    print("Predicted heights will be left in COLMAP units; ground-truth comparison")
    print("will be SKIPPED until you set this value.")
    print()
    print(f"To calibrate: measure the trunk height from base to first fork in metres")
    print(f"and set MEASURED_TRUNK_HEIGHT_M = <that value> above.")
    SCALE_M_PER_UNIT = None
else:
    SCALE_M_PER_UNIT = MEASURED_TRUNK_HEIGHT_M / trunk_extent_colmap
    print(f"Scale factor: 1 COLMAP unit = {SCALE_M_PER_UNIT:.4f} m")
    print(f"             (1 metre        = {1/SCALE_M_PER_UNIT:.3f} COLMAP units)")


## 3. Predicted branches in metres + save predictions CSV

If the scale factor was set in section 2, predictions are converted to metric.
Either way, the predictions are saved to a CSV so you can take them into the
field with you and measure the same branches with a tape.


In [ ]:
# Convert predicted heights to metres if we have a scale.
if SCALE_M_PER_UNIT is not None:
    preds_metric = [
        evaluate.PredBranch(
            branch_id=p.branch_id,
            attach_height_m=p.attach_height_m * SCALE_M_PER_UNIT,
            attach_angle_deg=p.attach_angle_deg,
        )
        for p in preds_colmap
    ]
else:
    preds_metric = preds_colmap   # values are still in COLMAP units; clearly mark in the CSV

import csv
predictions_csv = METRICS_DIR / f"{TREE_ID}_predictions_{RUN_LABEL}.csv"
unit_label = "m" if SCALE_M_PER_UNIT is not None else "colmap_units"
with open(predictions_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["branch_id", f"attach_height_{unit_label}", "attach_angle_deg"])
    for p in preds_metric:
        writer.writerow([p.branch_id, f"{p.attach_height_m:.3f}", f"{p.attach_angle_deg:.2f}"])
print(f"Wrote {len(preds_metric)} predictions → {predictions_csv}")

# Print a clean summary.
print()
print(f"{'id':>3}  {'height':>10}  {'angle':>8}")
for p in preds_metric:
    print(f"{p.branch_id:>3}  {p.attach_height_m:>8.2f} {unit_label}  {p.attach_angle_deg:>6.1f}°")


## 4. Ground-truth comparison (activates when GT CSV exists)

**Ground-truth CSV format** (`data/ground_truth/<tree_id>.csv`):

```
branch_id,attach_height_m,attach_angle_deg
0,1.20,42
1,1.85,28
2,2.40,55
...
```

Collect by tape-measuring each *visible primary* branch on the actual tree:
- `attach_height_m` — metres from the trunk base (ground level) to where the
  branch leaves the trunk
- `attach_angle_deg` — angle between the branch's initial direction and the
  trunk (90° = horizontal, 0° = parallel to trunk, 45° = typical for an oak)

`branch_id` is just an integer — doesn't need to match the predictions; the
matching is done by attachment height with a tolerance (default 30 cm).


In [ ]:
if not GT_CSV.exists():
    print(f"No ground-truth CSV at {GT_CSV} yet.")
    print()
    print("To enable scoring, create that file with the schema above and re-run this cell.")
    print(f"Suggestion: use the {len(preds_metric)} branches in the predictions CSV as a")
    print("starting checklist — go to the tree and tape-measure each one you can verify.")
    metrics = None
else:
    if SCALE_M_PER_UNIT is None:
        print("Ground-truth CSV exists, but no scale factor — skipping metric comparison.")
        print("Set MEASURED_TRUNK_HEIGHT_M in section 2 and re-run.")
        metrics = None
    else:
        gts = evaluate.load_ground_truth(GT_CSV)
        print(f"Loaded {len(gts)} ground-truth branches from {GT_CSV.name}.")
        print()
        print("Ground truth:")
        for g in gts:
            print(f"  id={g.branch_id:>2}  height={g.attach_height_m:>5.2f} m  angle={g.attach_angle_deg:>5.1f}°")

        metrics = evaluate.evaluate_tree(
            tree_id=TREE_ID,
            preds=preds_metric,
            gts=gts,
            height_tolerance_m=0.30,   # 30 cm matching window
        )
        print()
        print(f"=== Metrics ({TREE_ID}, run={RUN_LABEL}) ===")
        print(f"  GT branches:        {metrics.n_gt}")
        print(f"  Predicted branches: {metrics.n_pred}")
        print(f"  Matched pairs:      {metrics.n_matched}")
        print(f"  Recall:             {metrics.recall:.2%}")
        print(f"  Angle MAE:          {metrics.angle_mae_deg:.2f}°")
        print(f"  Height MAE:         {metrics.height_mae_m:.3f} m")


## 5. Persist metrics for the ablation table

Each run of this notebook (with a different `RUN_LABEL`) writes one row to
its own CSV in `outputs/metrics/`. The ablation cell at the bottom collates
all of them into a comparison table.


In [ ]:
if metrics is not None:
    evaluate.write_metrics_csv([metrics], OUT_CSV)
    print(f"Wrote metrics row → {OUT_CSV}")
else:
    print("Skipped — no metrics computed yet (need GT CSV + scale calibration).")


## 6. Visualise predicted vs. ground-truth branches

Side-by-side scatter: x = attachment height, y = attachment angle. Predicted
branches in green, ground-truth in red. Matched pairs are joined with a thin
grey line. Far-from-anything points are either misses (red, unmatched) or
false positives (green, unmatched).


In [ ]:
if metrics is None:
    print("Skipping visualisation — need both predictions and ground truth.")
else:
    pairs = evaluate.match_branches(preds_metric, gts, height_tolerance_m=0.30)
    matched_gt_ids  = {gi for gi, _ in pairs}
    matched_pred_ids = {pi for _, pi in pairs}

    fig, ax = plt.subplots(figsize=(9, 6))
    # ground truth
    gx = [g.attach_height_m  for g in gts]
    gy = [g.attach_angle_deg for g in gts]
    ax.scatter(gx, gy, c="tab:red",   s=80, marker="o", label=f"Ground truth (N={len(gts)})", zorder=2)
    # predicted
    px = [p.attach_height_m  for p in preds_metric]
    py = [p.attach_angle_deg for p in preds_metric]
    ax.scatter(px, py, c="tab:green", s=80, marker="x", label=f"Predicted (N={len(preds_metric)})", zorder=2)
    # match lines
    for gi, pi in pairs:
        ax.plot([gts[gi].attach_height_m,  preds_metric[pi].attach_height_m],
                [gts[gi].attach_angle_deg, preds_metric[pi].attach_angle_deg],
                c="gray", linewidth=0.8, alpha=0.6, zorder=1)

    ax.set_xlabel("Attachment height (m)")
    ax.set_ylabel("Attachment angle off trunk (°)")
    ax.set_title(f"{TREE_ID} — {RUN_LABEL} — recall {metrics.recall:.0%}, "
                  f"angle MAE {metrics.angle_mae_deg:.1f}°, height MAE {metrics.height_mae_m:.2f} m")
    ax.legend()
    ax.grid(alpha=0.3)
    viz.save_fig(fig, f"08_{TREE_ID}_{RUN_LABEL}_eval.png")
    plt.show()


## 7. Ablation table — collate every metrics CSV in `outputs/metrics/`

Each `<run_label>.csv` file is a row. To generate the proposal's three
ablations, re-run notebooks 06 + 07 + this notebook with different upstream
settings, changing `RUN_LABEL` in section 0 each time:

| Ablation | RUN_LABEL suggestion | Notebook setting to change |
|---|---|---|
| baseline (current) | `lab_filter_on` | unchanged |
| classical vs LAB segmentation | `classical_filter_on` | replace mask with `segmentation.classical_sky_mask` in nb 06 |
| two-view vs multi-view | `two_view` | use the cloud from nb 03 (`*_pair_*.npz`) instead of nb 04 |
| filter on vs off | `filter_off` | in nb 06, skip the filter — pass raw cloud through to nb 07 |

After each run, this cell shows the comparison table.


In [ ]:
import csv as _csv

rows = []
for csv_path in sorted(METRICS_DIR.glob("*.csv")):
    with open(csv_path) as f:
        reader = _csv.DictReader(f)
        for row in reader:
            row["run_label"] = csv_path.stem
            rows.append(row)

if not rows:
    print("No metrics CSVs in outputs/metrics/ yet.")
    print("Once you've scored at least one run (section 5 above), this cell will")
    print("show a comparison table across all run_labels.")
else:
    keys = ["run_label", "tree_id", "n_gt", "n_pred", "n_matched", "recall",
            "angle_mae_deg", "height_mae_m"]
    print(f"{'run_label':<22} {'tree_id':<14} {'n_gt':>4} {'n_pred':>6} {'matched':>7} "
          f"{'recall':>7} {'angle°':>8} {'height m':>10}")
    print("-" * 90)
    for r in rows:
        try:
            recall = float(r.get("recall", "nan"))
            ang    = float(r.get("angle_mae_deg", "nan"))
            hgt    = float(r.get("height_mae_m",  "nan"))
        except (TypeError, ValueError):
            recall = ang = hgt = float("nan")
        print(f"{r.get('run_label',''):<22} {r.get('tree_id',''):<14} "
              f"{r.get('n_gt','?'):>4} {r.get('n_pred','?'):>6} {r.get('n_matched','?'):>7} "
              f"{recall:>7.2%} {ang:>7.2f}° {hgt:>10.3f}")


## 8. Bar-chart summary (final-report figure)

Once you have ≥2 ablation runs saved to CSV, this cell renders the headline
bar chart: recall / angle MAE / height MAE per run, side by side. That figure
is what goes in the final report's "Results" section.


In [ ]:
if len(rows) < 2:
    print(f"Need at least 2 metrics CSVs to compare; currently have {len(rows)}.")
else:
    labels = [r["run_label"] for r in rows]
    try:
        recalls    = [float(r["recall"])         for r in rows]
        angle_maes = [float(r["angle_mae_deg"])  for r in rows]
        height_maes= [float(r["height_mae_m"])   for r in rows]
    except (KeyError, ValueError) as e:
        print(f"Couldn't parse rows for plotting: {e}")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].bar(labels, recalls,    color="tab:green");  axes[0].set_title("Recall (higher = better)");   axes[0].set_ylim(0, 1.0)
        axes[1].bar(labels, angle_maes, color="tab:orange"); axes[1].set_title("Angle MAE (deg, lower = better)")
        axes[2].bar(labels, height_maes,color="tab:blue");   axes[2].set_title("Height MAE (m, lower = better)")
        for ax in axes:
            ax.tick_params(axis="x", rotation=20)
            ax.grid(axis="y", alpha=0.3)
        fig.suptitle(f"Ablation comparison — {len(rows)} runs")
        viz.save_fig(fig, "08_ablation_summary.png")
        plt.show()
